In [1]:
%pip install -q "uniface[cpu]"

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 23.1/23.1 MB 34.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 160.4/160.4 kB 6.0 MB/s eta 0:00:00


In [2]:
import cv2
import matplotlib.pyplot as plt
from pathlib import Path

import uniface
from uniface.attribute import FaceAttribNet
from uniface.detection import RetinaFace

print(f"UniFace version: {uniface.__version__}")

UniFace version: 4.0.0


In [3]:
# Initialize face detector
detector = RetinaFace(confidence_threshold=0.5)

# Initialize face attribute model
face_attrib = FaceAttribNet()

print("Models initialized successfully!")

Attempt 1/3: 100%|██████████| 11.9M/11.9M [00:00<00:00, 94.7MB/s]
Attempt 1/3: 100%|██████████| 41.3M/41.3M [00:00<00:00, 137MB/s]


Models initialized successfully!


# 1. Process All Test Images
Each face gets a FaceStateResult with five independent probabilities: left_eye_open, right_eye_open, eyeglasses, mask, sunglasses.

These come from independent binary heads. They do not sum to 1 and several can be high at once (a face can wear both sunglasses and a mask). Threshold each attribute separately; never argmax

In [19]:
THRESHOLD = 0.5
CROP = 512

# the source photos differ in aspect ratio and framing, so crop each to the same square
SUBJECTS = [
    ('age_adult.jpg', 'eyes open'),
    ('mesh_face.jpg', 'no accessories'),
    ('state_b_glasses.jpg', 'glasses'),
    ('state_b_sunglasses.jpg', 'sunglasses'),
    ('state_b_mask.jpg', 'mask'),
]
demo_dir = Path('../assets/source')


def face_panel(path, margin=0.5):
    """Square crop centred on the largest face, so every panel is the same size."""
    image = cv2.imread(str(path))
    if image is None:
        print(f"Warning: Could not load image from {path}")
        return None, None
    faces = detector.detect(image)
    if not faces:
        return None, None
    face = max(faces, key=lambda f: f.bbox[2] - f.bbox[0])
    x1, y1, x2, y2 = face.bbox
    cx, cy = (x1 + x2) / 2, (y1 + y2) / 2
    h, w = image.shape[:2]
    half = min(max(x2 - x1, y2 - y1) * (1 + margin) / 2, w / 2, h / 2)
    cx = min(max(cx, half), w - half)
    cy = min(max(cy, half), h - half)
    crop = image[int(cy - half):int(cy + half), int(cx - half):int(cx + half)]
    crop = cv2.resize(crop, (CROP, CROP), interpolation=cv2.INTER_CUBIC)
    faces = detector.detect(crop)
    if not faces:
        return None, None
    return crop, face_attrib.predict(crop, max(faces, key=lambda f: f.bbox[2] - f.bbox[0]))


panels = []
for name, label in SUBJECTS:
    crop, result = face_panel(demo_dir / name)
    if crop is None:
        print(f'{name}: no face found')
        continue
    panels.append((cv2.cvtColor(crop, cv2.COLOR_BGR2RGB), result, label))
    print(f'{label:<12} {", ".join(result.labels(THRESHOLD)) or "none"}')

age_adult.jpg: no face found
mesh_face.jpg: no face found
state_b_glasses.jpg: no face found
state_b_sunglasses.jpg: no face found
state_b_mask.jpg: no face found


In [15]:
# After re-running cell 2JeKxmgmlS8L, let's inspect the panels list content.
if panels:
    print(f"Panels list contains {len(panels)} items.")
    for i, (crop, result, label) in enumerate(panels):
        print(f"Item {i+1}: Label='{label}', Crop Shape={crop.shape}, Result Attributes={result.as_dict().keys()}")
else:
    print("The panels list is still empty. Ensure images were downloaded and cell 2JeKxmgmlS8L was re-executed.")

The panels list is still empty. Ensure images were downloaded and cell 2JeKxmgmlS8L was re-executed.


In [5]:
if demo_dir.exists():
    print(f"Directory '{demo_dir}' exists.")
    print("Contents:")
    for item in demo_dir.iterdir():
        print(f"- {item.name}")
else:
    print(f"Directory '{demo_dir}' does not exist. Please ensure the 'assets/source' folder is correctly placed.")

Directory '../assets/source' does not exist. Please ensure the 'assets/source' folder is correctly placed.


# 2. Visualize Results
First row: Original images Second row: Faces annotated with the attributes above the threshold

In [22]:
if not panels:
    print("The 'panels' list is empty. No images to plot.")
else:
    fig, axes = plt.subplots(1, len(panels), figsize=(2.9 * len(panels), 4.6))

    # If there's only one panel, axes will not be an array, so make it one for consistent iteration
    if len(panels) == 1:
        axes = [axes]

    for ax, (crop, result, label) in zip(axes, panels):
        ax.imshow(crop)
        ax.set_title(label, fontsize=12)
        ax.axis('off')

        # the five heads are independent, so print every probability rather than a winner
        for i, (name, prob) in enumerate(result.as_dict().items()):
            on = prob > THRESHOLD
            ax.text(0.0, -0.05 - i * 0.075, name, transform=ax.transAxes, fontsize=9,
                    family='monospace', color='#111' if on else '#999', va='top')
            ax.text(1.0, -0.05 - i * 0.075, f'{"True " if on else "False"} {prob:.2f}',
                    transform=ax.transAxes, fontsize=9, family='monospace',
                    color='#0a7d3f' if on else '#999', va='top', ha='right')

    plt.tight_layout()
    plt.show()

The 'panels' list is empty. No images to plot.


In [27]:
import os
import requests

# Create the assets/source directory if it doesn't exist
os.makedirs(demo_dir, exist_ok=True)

# Define the base URL for the raw images
base_url = "https://raw.githubusercontent.com/uniface/uniface/main/assets/images/" # Corrected path

# Download each image
for name, _ in SUBJECTS:
    image_url = base_url + name
    image_path = demo_dir / name
    if not image_path.exists():
        print(f"Downloading {name}...")
        response = requests.get(image_url)
        response.raise_for_status() # Raise an exception for HTTP errors
        with open(image_path, 'wb') as f:
            f.write(response.content)
    else:
        print(f"{name} already exists.")

print("All images are now available in the assets/source directory.")

HTTPError: 404 Client Error: Not Found for url: https://raw.githubusercontent.com/uniface/uniface/main/assets/images/age_adult.jpg

In [28]:
import requests

def url_exists(url):
    """Checks if a URL exists and is accessible."""
    try:
        response = requests.head(url, allow_redirects=True, timeout=5)
        return response.status_code == 200
    except requests.exceptions.RequestException:
        return False

# Example usage with the problematic base_url and an image name
# This will likely return False given previous errors
example_image_name = 'age_adult.jpg'
full_example_url = base_url + example_image_name

if url_exists(full_example_url):
    print(f"URL '{full_example_url}' exists.")
else:
    print(f"URL '{full_example_url}' does not exist or is not accessible.")
    print("Please verify the base_url and image filenames in the GitHub repository.")

URL 'https://raw.githubusercontent.com/uniface/uniface/main/assets/images/age_adult.jpg' does not exist or is not accessible.
Please verify the base_url and image filenames in the GitHub repository.


After executing the above cell, please re-run the cell containing `panels = []` and the subsequent cells to process and visualize the images correctly.

In [21]:
import requests

repo_owner = 'uniface'
repo_name = 'uniface'
repo_path = 'assets/images'

api_url = f"https://api.github.com/repos/{repo_owner}/{repo_name}/contents/{repo_path}"

print(f"Fetching contents from GitHub API: {api_url}")
response = requests.get(api_url)

if response.status_code == 200:
    contents = response.json()
    print(f"Contents of '{repo_path}' in '{repo_owner}/{repo_name}':")
    for item in contents:
        if item['type'] == 'file':
            print(f"- {item['name']}")
else:
    print(f"Failed to fetch contents. Status Code: {response.status_code}")
    print(f"Response: {response.text}")

Fetching contents from GitHub API: https://api.github.com/repos/uniface/uniface/contents/assets/images
Failed to fetch contents. Status Code: 404
Response: {"message":"Not Found","documentation_url":"https://docs.github.com/rest/repos/contents#get-repository-content","status":"404"}


In [13]:
print('Please check the output of the previous cell to see the available image files in the GitHub repository. We will use this information to update the SUBJECTS list if necessary.')

Please check the output of the previous cell to see the available image files in the GitHub repository. We will use this information to update the SUBJECTS list if necessary.


In [24]:
import requests

repo_owner = 'uniface'
repo_name = 'uniface'

# Get the root contents of the repository
api_url = f"https://api.github.com/repos/{repo_owner}/{repo_name}/contents"

print(f"Fetching root contents from GitHub API: {api_url}")
response = requests.get(api_url)

if response.status_code == 200:
    contents = response.json()
    print(f"Contents of '{repo_owner}/{repo_name}' root directory:")
    for item in contents:
        print(f"- {item['type']}: {item['name']}")
else:
    print(f"Failed to fetch root contents. Status Code: {response.status_code}")
    print(f"Response: {response.text}")

Fetching root contents from GitHub API: https://api.github.com/repos/uniface/uniface/contents
Failed to fetch root contents. Status Code: 404
Response: {"message":"Not Found","documentation_url":"https://docs.github.com/rest/repos/contents#get-repository-content","status":"404"}


# 3. Inspect a Single Prediction
predict() returns a FaceStateResult and also writes the probabilities back onto the Face object (face.left_eye_open, face.right_eye_open, face.eyeglasses, face.mask, face.sunglasses)

In [25]:
image = cv2.imread(str(demo_dir / 'state_b_sunglasses.jpg'))

if image is None:
    print(f"Error: Could not load image from {demo_dir / 'state_b_sunglasses.jpg'}")
else:
    face = detector.detect(image)[0]

    result = face_attrib.predict(image, face)

    print(result)
    print()
    for name, prob in result.as_dict().items():
        marker = '✔' if prob > THRESHOLD else '✘'
        print(f"  {marker} {name} {prob:.4f}")

    print()
    print(f"Face enriched: sunglasses={face.sunglasses:.4f}, left_eye_open={face.left_eye_open:.4f}")

Error: Could not load image from ../assets/source/state_b_sunglasses.jpg


# Notes
The five values come from independent binary heads. They do not sum to 1, and more than one can be high at once, so threshold each separately and never use argmax.
# A face in sunglasses usually reads sunglasses true and both eyes false, because the lenses hide the eyes